In [16]:
import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path


from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error



In [17]:

# EDA from week2 Mod B
df_diabetes = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")
df_ckd = pd.read_csv("Chronic_Kidney_Dsease_data.csv")
df_hypertension = pd.read_csv("hypertension_dataset.csv")
df_alzheimers = pd.read_csv("alzheimers_disease_data.csv")

# Generate basic summaries
diabetes_desc = df_diabetes.describe(include='all')
ckd_desc = df_ckd.describe(include='all')
hypertension_desc = df_hypertension.describe(include='all')
alzheimers_desc = df_alzheimers.describe(include='all')

# Check for duplicates
diabetes_duplicates = df_diabetes.duplicated().sum()
ckd_duplicates = df_ckd.duplicated().sum()
alzheimers_duplicates = df_alzheimers.duplicated().sum()
_duplicates = df_hypertension.duplicated().sum()
# Check for nulls
diabetes_nulls = df_diabetes.isnull().sum()
ckd_nulls = df_ckd.isnull().sum()
hypertension_nulls = df_hypertension.isnull().sum()
alzheimers__nulls = df_alzheimers.isnull().sum()

# Display descriptive summaries

print("\n===alzheimers-Summary ===")
print(ckd_desc)
# Return duplicates and total null values per dataset
print("\n=== Duplicates in alzheimers_duplicates Dataset ===")
print(f"alzheimers: {alzheimers_duplicates}")

# Return nulls
print("\n=== Total Null Values in alzheimers Dataset ===")
print(f"Diabealzheimerstes: {alzheimers__nulls.sum()}")


#Diabetes Dataset
#Duplicates: 24,206 rows — significant duplication, should be reviewed or removed
#Missing values: None
#Usability: Usable after deduplication

#Chronic Kidney Disease (CKD) Dataset
#Duplicates: None
#Missing values: None\
#Usability: Clean and ready for analysis

#Hypertension Dataset
#Duplicates: None
#Missing values: None
#Usability: Ready to use

# Steps to clean up Diabetes Dataset 
#Remove duplicates from the diabetes dataset: df_diabetes.drop_duplicates(inplace=True)
#Consider class balance checks (e.g., ratio of positive to negative labels)
#Identify categorical features and apply encoding (pd.get_dummies or OrdinalEncoder)
#Explore mode, median, and outliers for inconsistent data (e.g., age = 0)

#1 Remove Duplicates
df_diabetes.drop_duplicates(inplace=True)
#2 Handling any missing values
df_diabetes.fillna(df_diabetes.median(), inplace=True)  # For numeric columns
df_diabetes.fillna("Unknown", inplace=True)    # For categorical columns
#3Check for Inconsistencies
  #Negative ages or values outside expected range
  #Incorrect data types (e.g., numeric coded as string)
#4 Check for Class Imbalance

print(df_diabetes['Diabetes_binary'].value_counts(normalize=True))
print(df_hypertension['Hypertension'].value_counts(normalize=True))
#print(df_ckd['classification'].value_counts(normalize=True)) 

# Encode Categorical Variables
# One-hot encoding (for logistic regression, tree-based models)
df = pd.get_dummies(df_diabetes, drop_first=True)

# Or ordinal encoding if there is a natural order
#print("Arun")
#print(df_ckd.columns)
categorical_cols = df_ckd.select_dtypes(include=['object', 'category']).columns.tolist()
#print("Categorical columns:", categorical_cols)

from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder()
df_ckd[['DoctorInCharge']] = encoder.fit_transform(df_ckd[['DoctorInCharge']])




===alzheimers-Summary ===
          PatientID          Age       Gender   Ethnicity  \
count   1659.000000  1659.000000  1659.000000  1659.00000   
unique          NaN          NaN          NaN         NaN   
top             NaN          NaN          NaN         NaN   
freq            NaN          NaN          NaN         NaN   
mean     830.000000    54.441230     0.515371     0.71308   
std      479.056364    20.549757     0.499914     1.00043   
min        1.000000    20.000000     0.000000     0.00000   
25%      415.500000    36.000000     0.000000     0.00000   
50%      830.000000    54.000000     1.000000     0.00000   
75%     1244.500000    72.000000     1.000000     1.00000   
max     1659.000000    90.000000     1.000000     3.00000   

        SocioeconomicStatus  EducationLevel          BMI      Smoking  \
count           1659.000000     1659.000000  1659.000000  1659.000000   
unique                  NaN             NaN          NaN          NaN   
top                  

In [18]:
target_col = "MMSE"  
df=df_alzheimers

In [22]:
# Perform Ridge, Lasso, and Elastic Net regressions on the uploaded dataset with MMSE as target.
assert target_col in df.columns, "MMSE not found in dataset."

# Drop obvious non-predictive identifiers / leakage
drop_cols = [c for c in ["PatientID", "DoctorInCharge"] if c in df.columns]
X = df.drop(columns=[target_col] + drop_cols, errors="ignore")
y = df[target_col]

# Identify column types
numeric_features = selector(dtype_include=np.number)(X)
categorical_features = selector(dtype_include=object)(X)

# Preprocess: impute and scale numeric; impute and one-hot categorical
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Define models with CV over alphas
ridge_alphas = np.logspace(-3, 3, 50)
ridge = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RidgeCV(alphas=ridge_alphas, cv=cv, scoring="neg_mean_squared_error"))
])

lasso = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LassoCV(cv=cv, random_state=42, n_alphas=100, max_iter=10000))
])

enet = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", ElasticNetCV(cv=cv, random_state=42,
                           l1_ratio=[.1,.3,.5,.7,.9,.95,.99,1.0],
                           n_alphas=100, max_iter=10000))
])

# Fit models
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
enet.fit(X_train, y_train)

# Predict
preds = {}
for name, model in [("Ridge", ridge), ("Lasso", lasso), ("ElasticNet", enet)]:
    preds[name] = {
        "y_pred_train": model.predict(X_train),
        "y_pred_test": model.predict(X_test),
        "estimator": model.named_steps["model"]
    }

def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)          # always returns MSE
    rmse = np.sqrt(mse)                                # take the square root manually
    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    return rmse, mae, r2

# Summarize metrics
rows = []
for name, d in preds.items():
    rmse_tr, mae_tr, r2_tr = metrics(y_train, d["y_pred_train"])
    rmse_te, mae_te, r2_te = metrics(y_test, d["y_pred_test"])
    # Alpha / l1_ratio
    if name == "Ridge":
        alpha = float(d["estimator"].alpha_)
        l1_ratio = 0.0
    elif name == "Lasso":
        alpha = float(d["estimator"].alpha_)
        l1_ratio = 1.0
    else:
        alpha = float(d["estimator"].alpha_)
        l1_ratio = float(d["estimator"].l1_ratio_)
    rows.append({
        "Model": name,
        "Alpha (λ)": alpha,
        "L1 Ratio": l1_ratio,
        "Train RMSE": rmse_tr,
        "Train MAE": mae_tr,
        "Train R²": r2_tr,
        "Test RMSE": rmse_te,
        "Test MAE": mae_te,
        "Test R²": r2_te
    })

metrics_df = pd.DataFrame(rows).sort_values("Test RMSE")

# Extract feature names after preprocessing
preprocess_only = preprocess.fit(X_train, y_train)
num_names = numeric_features
cat_names = list(preprocess_only.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)) if len(categorical_features)>0 else []
feature_names = np.array(list(num_names) + cat_names)

# Get coefficients
def get_coefs(pipeline, feature_names):
    # model is linear with coef_
    model = pipeline.named_steps["model"]
    # Ensure we pass through same preprocesser to get same columns ordering
    pre = pipeline.named_steps["preprocess"]
    # Create a dummy to determine transformed feature count
    # But we already computed feature_names from preprocess_only above (same spec)
    coefs = np.ravel(model.coef_)
    return pd.DataFrame({"feature": feature_names, "coef": coefs, "abs_coef": np.abs(coefs)}).sort_values("abs_coef", ascending=False)

ridge_coefs = get_coefs(ridge, feature_names)
lasso_coefs = get_coefs(lasso, feature_names)
enet_coefs = get_coefs(enet, feature_names)

# Count non-zero coefficients
nnz = {
    "Ridge": int((ridge_coefs["coef"]!=0).sum()),
    "Lasso": int((lasso_coefs["coef"]!=0).sum()),
    "ElasticNet": int((enet_coefs["coef"]!=0).sum())
}

# Prepare top coefficients tables
top_k = 15
ridge_top = ridge_coefs.head(top_k)
lasso_top = lasso_coefs.head(top_k)
enet_top = enet_coefs.head(top_k)

# Save outputs
metrics_path = "model_metrics.csv"
ridge_path = "ridge_top_coefs.csv"
lasso_path = "lasso_top_coefs.csv"
enet_path = "elasticnet_top_coefs.csv"

metrics_df.to_csv(metrics_path, index=False)
ridge_top.to_csv(ridge_path, index=False)
lasso_top.to_csv(lasso_path, index=False)
enet_top.to_csv(enet_path, index=False)

print("\n=== Regression Model Comparison (Ridge/Lasso/ElasticNet) ===")
print(metrics_df.round(4).to_string(index=False))

print("\n=== Top Ridge Coefficients ===")
print(ridge_top.round(4).to_string(index=False))

print("\n=== Top Lasso Coefficients ===")
print(lasso_top.round(4).to_string(index=False))

print("\n=== Top Elastic Net Coefficients ===")
print(enet_top.round(4).to_string(index=False))

# Also provide a compact summary with non-zero counts
summary_df = pd.DataFrame([
    {"Model": "Ridge", "Selected (non-zero)": nnz["Ridge"], "Alpha (λ)": rows[0]["Alpha (λ)"]},
    {"Model": "Lasso", "Selected (non-zero)": nnz["Lasso"], "Alpha (λ)": rows[1]["Alpha (λ)"]},
    {"Model": "ElasticNet", "Selected (non-zero)": nnz["ElasticNet"], "Alpha (λ)": rows[2]["Alpha (λ)"]},
])
summary_df

/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1613: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1613: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(



=== Regression Model Comparison (Ridge/Lasso/ElasticNet) ===
     Model  Alpha (λ)  L1 Ratio  Train RMSE  Train MAE  Train R²  Test RMSE  Test MAE  Test R²
     Lasso     0.1809       1.0      8.2886     7.2451    0.0790     8.0393    6.9778   0.1070
ElasticNet     0.1809       1.0      8.2886     7.2451    0.0790     8.0393    6.9778   0.1070
     Ridge   104.8113       0.0      8.2459     7.1772    0.0885     8.0564    6.9525   0.1032

=== Top Ridge Coefficients ===
                 feature    coef  abs_coef
               Diagnosis -2.7499    2.7499
        MemoryComplaints  0.8747    0.8747
                     ADL -0.8279    0.8279
    FunctionalAssessment -0.7402    0.7402
      BehavioralProblems  0.6791    0.6791
   CardiovascularDisease  0.2836    0.2836
                Diabetes -0.2595    0.2595
          Disorientation  0.2356    0.2356
             DiastolicBP -0.2270    0.2270
            Hypertension  0.1980    0.1980
CholesterolTriglycerides  0.1941    0.1941
          

,Model,Selected (non-zero),Alpha (λ)
0,Ridge,32,104.811313
1,Lasso,12,0.180931
2,ElasticNet,12,0.180931


In [26]:
# Create simple matplotlib charts (no specified colors) and save as PNGs
import matplotlib.pyplot as plt
import pandas as pd

# Reload metrics for plotting
metrics_df = pd.read_csv("model_metrics.csv")

# Plot Test R^2 comparison
plt.figure()
plt.bar(metrics_df["Model"], metrics_df["Test R²"])
plt.title("Test R² by Model")
plt.xlabel("Model")
plt.ylabel("Test R²")
plt.tight_layout()
plt.savefig("test_r2_by_model.png")
plt.close()

# Plot top Lasso coefficients by absolute value
lasso_top = pd.read_csv("lasso_top_coefs.csv")
plt.figure()
plt.barh(lasso_top["feature"].head(15)[::-1], lasso_top["abs_coef"].head(15)[::-1])
plt.title("Top 15 Lasso Coefficients (|coef|)")
plt.xlabel("|Coefficient|")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("top15_lasso_coeffs.png")
plt.close()

"test_r2_by_model.png", "top15_lasso_coeffs.png"


('test_r2_by_model.png', 'top15_lasso_coeffs.png')